## Setup

In [1]:
import os
import json
import itertools
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import drive

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
ROOT       = "/content/drive/MyDrive/chest_xray_subset"
LABELS_CSV = f"{ROOT}/subset_labels_v2.csv"
BBOX_CSV   = f"{ROOT}/BBox_List_2017.csv"
IMAGE_DIR  = f"{ROOT}/images"
SPLIT_DIR  = f"{ROOT}/splits"

IMG_COL = "Image Index"
PID_COL = "Patient ID"

# Model outputs are bound to this order. Changing it breaks every saved checkpoint.
LABELS = [
    "Atelectasis", "Consolidation", "Infiltration", "Pneumothorax",
    "Edema", "Emphysema", "Fibrosis", "Effusion",
    "Pneumonia", "Pleural_Thickening", "Cardiomegaly", "Nodule",
    "Mass", "Hernia",
]
SEED        = 42
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

print("CONFIG OK ->", ROOT)

CONFIG OK -> /content/drive/MyDrive/chest_xray_subset


## 1. Load the subset

`subset_labels_v2.csv` is our 31,000-image subset plus 77 extra annotated images we pulled from the full dataset later (see the appendix at the end).


In [4]:
df = pd.read_csv(LABELS_CSV)
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])

print(f"images  : {len(df):,}")
print(f"patients: {df[PID_COL].nunique():,}")
print(f"nulls   : {df.isna().sum().sum()}")

print("\nclass prevalence:")
for label in LABELS:
    n = int(df[label].sum())
    print(f"   {label:<20} {n:>6,}   {n/len(df):>6.2%}")

images  : 31,077
patients: 11,907
nulls   : 0

class prevalence:
   Atelectasis           5,130   16.51%
   Consolidation         3,083    9.92%
   Infiltration          7,276   23.41%
   Pneumothorax          3,337   10.74%
   Edema                 2,303    7.41%
   Emphysema             2,516    8.10%
   Fibrosis              1,686    5.43%
   Effusion              6,640   21.37%
   Pneumonia             1,431    4.60%
   Pleural_Thickening    2,729    8.78%
   Cardiomegaly          2,776    8.93%
   Nodule                3,464   11.15%
   Mass                  3,450   11.10%
   Hernia                  227    0.73%


## 2. Stratification key

We want each split to contain a similar share of every disease. The problem is that one image can carry several diseases at once, so there is no single class to stratify on.

Our approach: give each patient one key — the **rarest** disease they have. A patient with Effusion and Hernia is filed under Hernia. This makes sure the rare classes get spread evenly, which is what we actually care about. Hernia only has 227 images in the entire dataset, so a random split could easily leave the test set with almost none.

In [5]:
patient = df.groupby(PID_COL)[LABELS].max()
rarity_order = sorted(LABELS, key=lambda l: df[l].sum())

def rarest_label(row):
    for label in rarity_order:
        if row[label] == 1:
            return label
    return "No_Finding"

patient["strat_key"] = patient.apply(rarest_label, axis=1)

print(f"patients: {len(patient):,}")
print(f"smallest group: {patient['strat_key'].value_counts().min()} patients")

patients: 11,907
smallest group: 134 patients


## 3. Build the splits

Order matters here. We remove the annotated patients **before** splitting rather than moving them into `test` afterwards.

Why not just put them in `test`? Because they are not a random sample. NIH picked clear cases of eight specific diseases to annotate, so all of them are positive and none are healthy. Dropping that group into the test set would make the test set unrepresentative and push our classification scores up artificially. We would be trading one problem for another.

In [6]:
bbox = pd.read_csv(BBOX_CSV)
bbox.columns = [c.strip() for c in bbox.columns]

bbox_imgs = bbox[bbox[IMG_COL].isin(df[IMG_COL])][IMG_COL]
BBOX_PIDS = set(df[df[IMG_COL].isin(bbox_imgs)][PID_COL])
pool = patient[~patient.index.isin(BBOX_PIDS)]

print(f"annotated images we have  : {bbox_imgs.nunique()} / {bbox[IMG_COL].nunique()}")
print(f"held out for localization : {len(BBOX_PIDS):,} patients")
print(f"pool for classification   : {len(pool):,} patients")

annotated images we have  : 670 / 880
held out for localization : 578 patients
pool for classification   : 11,329 patients


In [7]:
train_ids, temp_ids, _, temp_strat = train_test_split(
    pool.index.to_numpy(), pool["strat_key"].to_numpy(),
    test_size=VAL_RATIO + TEST_RATIO,
    stratify=pool["strat_key"].to_numpy(), random_state=SEED,
)

val_ids, test_ids = train_test_split(
    temp_ids, test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
    stratify=temp_strat, random_state=SEED,
)

# Half to pick the Grad-CAM threshold, half to report on. Doing both on the same
# images would mean choosing whichever threshold makes our number look best.
loc_ids = np.array(sorted(BBOX_PIDS))
np.random.default_rng(SEED).shuffle(loc_ids)
half = len(loc_ids) // 2

def subset_of(ids):
    return df[df[PID_COL].isin(ids)].reset_index(drop=True)

splits = {
    "train":      subset_of(train_ids),
    "val":        subset_of(val_ids),
    "test":       subset_of(test_ids),
    "loc_tune":   subset_of(loc_ids[:half]),
    "loc_report": subset_of(loc_ids[half:]),
}

print(f"{'split':<12}{'images':>9}{'patients':>11}")
for name, part in splits.items():
    print(f"{name:<12}{len(part):>9,}{part[PID_COL].nunique():>11,}")

split          images   patients
train          18,013      7,930
val             3,978      1,699
test            3,904      1,700
loc_tune        2,606        289
loc_report      2,576        289


## 4. Validation

Every pair of splits must share zero patients. If anything shows FAIL, stop — all later results would be invalid.

In [8]:
print("=== PATIENT LEAKAGE ===")
for a, b in itertools.combinations(splits, 2):
    n = len(set(splits[a][PID_COL]) & set(splits[b][PID_COL]))
    print(f"{a:<11} vs {b:<11}: {n:>3} shared   {'OK' if n == 0 else 'FAIL'}")

clf = {k: splits[k] for k in ("train", "val", "test")}

print("\n=== PREVALENCE ===")
print(f"{'label':<20}{'train':>9}{'val':>9}{'test':>9}{'dev':>9}")
for label in LABELS:
    r = {k: p[label].mean() for k, p in clf.items()}
    print(f"{label:<20}" + "".join(f"{r[k]:>9.4f}" for k in clf)
          + f"{max(r.values())-min(r.values()):>9.4f}")

print("\n=== POSITIVE COUNTS ===")
print(f"{'label':<20}{'train':>9}{'val':>9}{'test':>9}")
for label in LABELS:
    c = {k: int(p[label].sum()) for k, p in clf.items()}
    print(f"{label:<20}" + "".join(f"{c[k]:>9,}" for k in clf)
          + ("  <-- few positives" if c["test"] < 50 else ""))

=== PATIENT LEAKAGE ===
train       vs val        :   0 shared   OK
train       vs test       :   0 shared   OK
train       vs loc_tune   :   0 shared   OK
train       vs loc_report :   0 shared   OK
val         vs test       :   0 shared   OK
val         vs loc_tune   :   0 shared   OK
val         vs loc_report :   0 shared   OK
test        vs loc_tune   :   0 shared   OK
test        vs loc_report :   0 shared   OK
loc_tune    vs loc_report :   0 shared   OK

=== PREVALENCE ===
label                   train      val     test      dev
Atelectasis            0.1582   0.1594   0.1568   0.0026
Consolidation          0.0909   0.0945   0.0853   0.0092
Infiltration           0.2193   0.2142   0.2147   0.0051
Pneumothorax           0.0892   0.0980   0.0961   0.0088
Edema                  0.0697   0.0729   0.0794   0.0097
Emphysema              0.0741   0.0822   0.0973   0.0233
Fibrosis               0.0587   0.0553   0.0543   0.0044
Effusion               0.1972   0.1976   0.1972   0.0004
Pne

## 5. Save

In [9]:
os.makedirs(SPLIT_DIR, exist_ok=True)

for name, part in splits.items():
    part.to_csv(f"{SPLIT_DIR}/{name}.csv", index=False)
    print(f"{name:<12} {len(part):>7,} rows saved")

meta = {
    "seed": SEED,
    "ratios": {"train": TRAIN_RATIO, "val": VAL_RATIO, "test": TEST_RATIO},
    "source_csv": os.path.basename(LABELS_CSV),
    "total_images": len(df),
    "total_patients": int(df[PID_COL].nunique()),
    "bbox_coverage": f"{bbox_imgs.nunique()}/{bbox[IMG_COL].nunique()}",
    "label_order": LABELS,
    "method": "bbox patients held out first; remaining pool split by patient, "
              "stratified on each patient's rarest condition",
    "splits": {k: {"images": len(v), "patients": int(v[PID_COL].nunique())}
               for k, v in splits.items()},
}

with open(f"{SPLIT_DIR}/split_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

print("\nmetadata saved")

train         18,013 rows saved
val            3,978 rows saved
test           3,904 rows saved
loc_tune       2,606 rows saved
loc_report     2,576 rows saved

metadata saved


In [10]:
for name in splits:
    p = f"{SPLIT_DIR}/{name}.csv"
    print(f"{name:<12}{os.path.getsize(p)/1e6:>7.2f} MB   {len(pd.read_csv(p)):>7,} rows")

train          1.85 MB    18,013 rows
val            0.41 MB     3,978 rows
test           0.40 MB     3,904 rows
loc_tune       0.28 MB     2,606 rows
loc_report     0.28 MB     2,576 rows


## Notes for the next stage

**Grad-CAM is evaluated on the annotated images only.** `loc_tune` and `loc_report` hold about 5,200 images between them, but only 670 of those actually have boxes. The rest are other X-rays belonging to the same patients, kept out of training so nothing leaks. Filter against `BBox_List_2017.csv` before computing any localization score.

**Use IoBB, not IoU.** Grad-CAM produces a spread-out heatmap rather than a tight box, so IoU gives near-zero scores that look like failure but are not. The NIH paper reports IoBB at thresholds of 0.1, 0.25 and 0.5.

**Boxes exist for 8 diseases only** — Atelectasis, Cardiomegaly, Effusion, Infiltration, Mass, Nodule, Pneumonia, Pneumothorax. The other six spread across the lung, so pointing at one spot does not mean anything for them.

**Hernia is at the limit.** 28 positives in val and 30 in test is all the dataset allows. Report its AUROC with a bootstrap confidence interval and mention it as a limitation.

**`No_Finding` is not a model output.** We keep the column for filtering and statistics, but the classifier has exactly 14 sigmoid outputs. "No finding" simply means all 14 are negative.

**Loading images from Drive during training is slow.** Every file goes over the network. Zip the `images/` folder and unzip it once to `/content` when a training session starts.

---

### Appendix — where the 77 extra images came from

Our original subset of 31,000 images happened to include 593 of the 880 annotated images. We pulled 77 more from the full Kaggle dataset one file at a time and added them to `subset_labels_v2.csv`, bringing coverage to 670 of 880 (76%).

That download needed a personal Kaggle API token, so the code is not included here. It is a one-time step and the result is already saved in Drive.